<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="cognitiveclass.ai logo">
</center>


# **Introduction to Model APIs in Keras**


In this lab, we will use a simple example to introduce you to different ways of creating models in Keras. 


## __Table of Contents__

1. [Objectives](#toc-objectives)
2. [Setup](#toc-setup)
    1. [Installing Required Libraries](#toc-installing-required-libraries)
    2. [Importing Required Libraries](#toc-importing-required-libraries)
3. [Types of Model APIs in Keras](#toc-types-of-model-apis-in-keras)
    1. [Task Definition](#toc-task-definition)
    2. [The Sequential Model API](#toc-the-sequential-model-api)
    3. [Create the Sequential Model](#toc-create-the-sequential-model)
    4. [Train and Evaluate the Model](#toc-train-and-evaluate-the-model)
    5. [The Functional Model API](#toc-the-functional-model-api)
    6. [Model Subclassing](#toc-model-subclassing)


## **Objectives** <a id="toc-objectives"></a>

After completing this lab you will be able to:

- __Understand__ different use cases for the Sequential and Functional APIs
- __Build__ custom models using sub-classing in Keras


## **Setup** <a id="toc-setup"></a>


For this lab, we will be using the following libraries:

*   [`pandas`](https://pandas.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for managing the data.
*   [`numpy`](https://numpy.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for mathematical operations.
*   [`sklearn`](https://scikit-learn.org/stable/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for machine learning and machine-learning-pipeline related functions.
*   [`seaborn`](https://seaborn.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for visualizing the data.
*   [`matplotlib`](https://matplotlib.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for additional plotting tools.
*   [`keras`](https://keras.io/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) for loading datasets.


### **Installing Required Libraries** <a id="toc-installing-required-libraries"></a>

The following required libraries are pre-installed in the Skills Network Labs environment. However, if you run these notebook commands in a different Jupyter environment (like Watson Studio or Ananconda), you will need to install these libraries by removing the `#` sign before `!pip install mlxtend` in the following code cell.


The following required libraries are __not__ pre-installed in the Skills Network Labs environment. __You will need to run the following cell__ to install them:


In [87]:
%%capture

#!pip install mlxtend
#!pip install --upgrade tensorflow

### **Importing Required Libraries** <a id="toc-importing-required-libraries"></a>


In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import keras
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

from sklearn import metrics, preprocessing
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import shuffle

print(tf.__version__)

seed = 7
np.random.seed(seed)

2.21.0


## **Types of Model APIs in Keras** <a id="toc-types-of-model-apis-in-keras"></a>


There are three main ways of creating models in Keras.


* **The Sequential Model**: This is a straightforward way of stacking layers but is limited to having a single-input and single-output stack of layers. 
* **The Functional API**: This can support arbitrary model architectures, and is more flexible in enabling complex implementations.

* **Model subclassing**: This is a way of implementing models from scratch. This is mainly used for out-of-the-box research use cases.


In the rest of the lab, we will go through all of these ways of creating models, and walk through a use case for implementing and training each of these model architectures in Keras and Tensorflow. 


### __Task Definition__ <a id="toc-task-definition"></a>


In this lab, we will be performing a simple classification task using the 'Sonar' dataset from [UCI](http://archive.ics.uci.edu/ml/datasets/connectionist+bench+\(sonar,+mines+vs.+rocks\)?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkML311Coursera747-2022-01-01).

The file contains sonar signal patterns collected by bouncing sound waves off either a metal cylinder or rocks under different environmental conditions and viewing angles. The transmitted sonar signal is a frequency-modulated chirp that gradually increases in frequency. The dataset includes signals recorded from a wide range of aspect angles, covering 90 degrees for the metal cylinder and 180 degrees for the rocks.

Each pattern consists of 60 numerical values ranging from 0.0 to 1.0. Each value represents the energy measured within a specific frequency band over a certain time interval. Higher frequency bands are integrated later in time because those frequencies are transmitted later during the chirp signal.

> The label for each record is **R** if the object is a rock and **M** if the object is a mine (metal cylinder). The records are ordered according to increasing aspect angle, although the labels themselves do not directly encode the angle information.



Let's start by reading in the dataset and defining our feature and target variables.


In [ ]:
dataframe = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML311-Coursera/labs/Module1/L1/data/sonar.csv",
    header=None,
)

In [90]:
dataframe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 208 entries, 0 to 207
Data columns (total 61 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       208 non-null    float64
 1   1       208 non-null    float64
 2   2       208 non-null    float64
 3   3       208 non-null    float64
 4   4       208 non-null    float64
 5   5       208 non-null    float64
 6   6       208 non-null    float64
 7   7       208 non-null    float64
 8   8       208 non-null    float64
 9   9       208 non-null    float64
 10  10      208 non-null    float64
 11  11      208 non-null    float64
 12  12      208 non-null    float64
 13  13      208 non-null    float64
 14  14      208 non-null    float64
 15  15      208 non-null    float64
 16  16      208 non-null    float64
 17  17      208 non-null    float64
 18  18      208 non-null    float64
 19  19      208 non-null    float64
 20  20      208 non-null    float64
 21  21      208 non-null    float64
 22  22

In [ ]:
dataframe.describe(include="all")

,0,1,2,3,4,5,6,7,8,9,...,51,52,53,54,55,56,57,58,59,60
count,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,...,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208.000000,208
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,111
mean,0.029164,0.038437,0.043832,0.053892,0.075202,0.104570,0.121747,0.134799,0.178003,0.208259,...,0.013420,0.010709,0.010941,0.009290,0.008222,0.007820,0.007949,0.007941,0.006507,NaN
std,0.022991,0.032960,0.038428,0.046528,0.055552,0.059105,0.061788,0.085152,0.118387,0.134416,...,0.009634,0.007060,0.007301,0.007088,0.005736,0.005785,0.006470,0.006181,0.005031,NaN
min,0.001500,0.000600,0.001500,0.005800,0.006700,0.010200,0.003300,0.005500,0.007500,0.011300,...,0.000800,0.000500,0.001000,0.000600,0.000400,0.000300,0.000300,0.000100,0.000600,NaN
25%,0.013350,0.016450,0.018950,0.024375,0.038050,0.067025,0.080900,0.080425,0.097025,0.111275,...,0.007275,0.005075,0.005375,0.004150,0.004400,0.003700,0.003600,0.003675,0.003100,NaN
50%,0.022800,0.030800,0.034300,0.044050,0.062500,0.092150,0.106950,0.112100,0.152250,0.182400,...,0.011400,0.009550,0.009300,0.007500,0.006850,0.005950,0.005800,0.006400,0.005300,NaN
75%,0.035550,0.047950,0.057950,0.064500,0.100275,0.134125,0.154000,0.169600,0.233425,0.268700,...,0.016725,0.014900,0.014500,0.012100,0.010575,0.010425,0.010350,0.010325,0.008525,NaN


In [ ]:
dataset = dataframe.values

X = dataset[:, 0:60].astype(float)
y = dataset[:, 60]

# encode textual labels as integers
le = LabelEncoder()  #
encoded_y = le.fit_transform(y)
print(encoded_y)

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


Now, we will split the dataset into a training and testing set with a 80:20 ratio.


In [93]:
X_train, X_test, y_train, y_test = train_test_split(
    X, encoded_y, test_size=0.20, random_state=42
)

### **The Sequential Model API** <a id="toc-the-sequential-model-api"></a>


The Sequential API groups a linear stack of layers into a `tf.keras.Model`. It is designed for simple neural networks where data flows through the layers in a single sequence. However, it cannot create models with shared layers, branching structures, multiple inputs, or multiple outputs. Despite these limitations, it provides convenient tools for model training and inference.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML311-Coursera/labs/Module1/L1/images/sequential_model.png" alt="The Sequential model API" width="50%">


As its name suggests, the Sequential API is simple and easy to use, making it a good choice for straightforward deep learning tasks. Later, you will explore the Functional API, which offers greater flexibility and supports more complex model architectures.

### **Create the Sequential Model** <a id="toc-create-the-sequential-model"></a>


To create a Sequential model, we first initialize it as an instance of the Sequential class:
```python
model = Sequential()
```
We then build the network by adding layers one at a time using:
```python
model.add(layer)
```
This method can add different types of layers, such as dense layers, convolutional layers, normalization layers, activation functions, and softmax classifiers. Similarly, the last layer can be removed using:
```python
model.pop()
```
In most cases, the first layer includes an `input_shape` argument. This is because Keras determines the shape of the model weights based on the shape of the input data. The weights are created automatically the first time the model receives input data. After the input shape is known, later layers can infer their input dimensions automatically. The general guidelines for choosing the input shape are described in the *5.3.1_Optimizer_Lab* notebook.

The Sequential API is best suited for linear networks. If a model contains non-linear connections, multiple inputs, multiple outputs, or shared layers, the Functional API is a more appropriate choice.

In [94]:
def baseline_model():
    model = Sequential()
    model.add(Dense(60, activation="relu", input_shape=(60,)))
    model.add(Dense(60, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))
    model.compile(
        optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
    )

    return model

In [95]:
estimator = baseline_model()
estimator.summary()

/home/wusu/anaconda3/envs/Python_3_10/lib/python3.10/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_33 (Dense)                │ (None, 60)             │         3,660 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 60)             │         3,660 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 1)              │            61 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,381 (28.83 KB)

 Trainable params: 7,381 (28.83 KB)

 Non-trainable params: 0 (0.00 B)

### **Train and Evaluate the Model** <a id="toc-train-and-evaluate-the-model"></a>


After creating the model, compiling it with your choice of optimizer and loss function, and doing a sanity check on its contents, you are now ready to build.

Simply call the `.fit()` to train the model. 


In [96]:
# Write your solution here
history = estimator.fit(
    X_train, y_train, epochs=20, batch_size=16, validation_data=(X_test, y_test)
)

Epoch 1/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4699 - loss: 0.6957 - val_accuracy: 0.6190 - val_loss: 0.6665
Epoch 2/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5783 - loss: 0.6814 - val_accuracy: 0.7143 - val_loss: 0.6578
Epoch 3/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6807 - loss: 0.6668 - val_accuracy: 0.7381 - val_loss: 0.6513
Epoch 4/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6446 - loss: 0.6539 - val_accuracy: 0.7857 - val_loss: 0.6216
Epoch 5/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6265 - loss: 0.6426 - val_accuracy: 0.8095 - val_loss: 0.6071
Epoch 6/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6988 - loss: 0.6309 - val_accuracy: 0.7143 - val_loss: 0.6143
Epoch 7/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7108 - loss: 0.6178 - val_accuracy: 0.8333 - val_loss: 0.5719
Epoch 8/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7530 - loss: 0.5988 - val_accuracy: 0.7619 - val_loss:

<details>
    <summary>Click here for Solution</summary>

```python
estimator.fit(X_train, y_train, epochs=10, batch_size=16)
```

</details>


After that completes, just use `.predict()` to evaluate against your test set. In this case, the `accuracy` will be used as the metric.


In [97]:
y_pred = estimator.predict(X_test)
y_pred = [1 if x >= 0.5 else 0 for x in y_pred]
metrics.accuracy_score(y_pred, y_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


0.8571428571428571

### **The Functional Model API** <a id="toc-the-functional-model-api"></a>


The `Sequential` class in Keras is commonly used for building simple neural network architectures where layers are arranged in a single linear stack. In contrast, the `Functional` API is preferred by many deep learning practitioners because it is more flexible and supports more complex model designs.

The Functional API allows you to create:

- non-linear network topologies
- multiple inputs and outputs
- shared layers
- branching architectures

In a Functional model, layers are connected as a graph rather than a simple sequence, allowing data to flow through the network in multiple ways.
```python
inputs = Input(shape=(...))
branch_1 = Conv2D(...)(inputs)
branch_2 = MaxPooling2D(...)(inputs)
outputs = Concatenate()([branch_1, branch_2])
model = Model(inputs=inputs, outputs=outputs)
```


<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML311-Coursera/labs/Module1/L1/images/functional_model.png" alt="The Sequential model API" width="50%">


Any model that can be built using `Sequential()` can also be implemented using the Functional API. Many modern deep learning architectures, such as ResNet, GoogLeNet, and Xception, are implemented using the Functional API because of its flexibility.

Both `Sequential` and the `Functional` API use the same methods for training, evaluation, and inference. The Model class provides:

- `.fit()` for training
- `.evaluate()` for model evaluation
- `.predict()` for inference

The following example demonstrates how to train and evaluate a model using the MNIST dataset.

We will start by loading the dataset directly using Keras.


In [98]:
def functional_model():
    inputs = keras.Input(shape=(60,))
    layer1 = Dense(60, activation="relu")(inputs)
    layer2 = Dense(60, activation="relu")(layer1)
    outputs = Dense(1, activation="sigmoid")(layer2)

    model = keras.Model(inputs, outputs)

    # Compile model, write code below
    model.compile(
        optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
    )

    return model

In [99]:
functional_estimator = functional_model()
estimator.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_33 (Dense)                │ (None, 60)             │         3,660 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_34 (Dense)                │ (None, 60)             │         3,660 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_35 (Dense)                │ (None, 1)              │            61 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,145 (86.51 KB)

 Trainable params: 7,381 (28.83 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 14,764 (57.68 KB)

In [ ]:
functional_estimator.fit(
    X_train, y_train, epochs=20, batch_size=16, validation_data=(X_test, y_test)
)

Epoch 1/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5723 - loss: 0.6740 - val_accuracy: 0.8095 - val_loss: 0.6101
Epoch 2/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6747 - loss: 0.6519 - val_accuracy: 0.7619 - val_loss: 0.6133
Epoch 3/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6988 - loss: 0.6353 - val_accuracy: 0.8095 - val_loss: 0.5876
Epoch 4/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6747 - loss: 0.6193 - val_accuracy: 0.8333 - val_loss: 0.5503
Epoch 5/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6747 - loss: 0.5987 - val_accuracy: 0.8571 - val_loss: 0.5419
Epoch 6/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7590 - loss: 0.5833 - val_accuracy: 0.8571 - val_loss: 0.5179
Epoch 7/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7289 - loss: 0.5581 - val_accuracy: 0.8333 - val_loss: 0.4811
Epoch 8/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7892 - loss: 0.5421 - val_accuracy: 0.8571 - val_loss:

Similar to what we did before, evaluate the estimator's performance on the test dataset, and print out the accuracy.


In [101]:
# Write your solution here
y_pred = functional_estimator.predict(X_test)
y_pred = [1 if x >= 0.5 else 0 for x in y_pred]
metrics.accuracy_score(y_pred, y_test)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


0.8809523809523809

### **Model Subclassing** <a id="toc-model-subclassing"></a>


Next, we will learn how to make new models and classes using model sub-classing. This method is more flexible and can be used to implement out-of-box models, but this comes at a cost, it is a lot harder to utilize than `Sequential()` and `Functional()` API. 

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML311-Coursera/labs/Module1/L1/images/subclassing.png" alt="The Sequential model API" width="30%">

In model subclassing, we create a custom class by inheriting from `tf.keras.Model`. The layers are usually defined inside the `__init__()` method, while the forward computation is implemented in the `call()` method.

This approach allows complete control over how data flows through the network and supports:

- custom layer behavior
- dynamic architectures
- multiple inputs and outputs
- conditional logic inside the model
- custom convolution or pooling operations

In the following example, we will implement a custom model using subclassing. Although the architecture behaves similarly to a Sequential model, it will be defined manually and will support multiple inputs and outputs.

In [102]:
class MyModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.dense1 = Dense(60, activation="relu")
        self.dense2 = Dense(60, activation="relu")
        self.dense3 = Dense(1, activation="sigmoid")

    def call(self, inputs):
        x = self.dense1(inputs)
        x = self.dense2(x)
        return self.dense3(x)


def subclass_model():
    inputs = keras.Input(shape=(60,))
    mymodel = MyModel()
    outputs = mymodel.call(inputs)

    model = keras.Model(inputs, outputs)
    model.compile(
        optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
    )

    return model

In [ ]:
subclass_estimator = subclass_model()
subclass_estimator.fit(
    X_train, y_train, epochs=20, batch_size=16, validation_data=(X_test, y_test)
)

Epoch 1/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5241 - loss: 0.6973 - val_accuracy: 0.6905 - val_loss: 0.6702
Epoch 2/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5663 - loss: 0.6665 - val_accuracy: 0.6190 - val_loss: 0.6345
Epoch 3/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6566 - loss: 0.6513 - val_accuracy: 0.7381 - val_loss: 0.6353
Epoch 4/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7048 - loss: 0.6376 - val_accuracy: 0.7857 - val_loss: 0.6211
Epoch 5/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7229 - loss: 0.6208 - val_accuracy: 0.8333 - val_loss: 0.5884
Epoch 6/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7590 - loss: 0.6061 - val_accuracy: 0.8333 - val_loss: 0.5718
Epoch 7/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7590 - loss: 0.5874 - val_accuracy: 0.8571 - val_loss: 0.5373
Epoch 8/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7590 - loss: 0.5681 - val_accuracy: 0.8810 - val_loss:

In [104]:
y_pred = subclass_estimator.predict(X)
y_pred = [1 if x >= 0.5 else 0 for x in y_pred]
metrics.accuracy_score(y_pred, encoded_y)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


0.8653846153846154

## **References**


* Sequential Model: https://www.tensorflow.org/guide/keras/sequential_model

* Functional Model: https://www.tensorflow.org/guide/keras/functional


## **Authors**


[Kopal Garg](https://www.linkedin.com/in/gargkopal/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkML311Coursera35714171-2022-01-01)

Kopal Garg is a Masters student in computer science at the University of Toronto, and a recent Biomedical Engineer from the University of Waterloo. 

Su Wu

## **Change Log**


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2022-08-17|0.1|Kopal Garg|Created Lab|
|2022-09-08|0.1|Steve Hord|QA pass edits|
| 2026-05-24     | 0.1     | Su Wu| Documentation and script edit    |

Copyright © 2022 IBM Corporation. All rights reserved.
